# Probabilistic Modelling Case Studies

This notebook demonstrates three compact, reproducible approaches to reasoning under uncertainty: nonlinear response modelling, Bayesian updating, and sample-size planning for an imperfect screening test.

All values are synthetic and chosen for illustration. The focus is on transparent assumptions, reusable functions, and communicating limitations.

## Environment and plotting defaults

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(42)

## 1. Modelling diminishing returns

Suppose a product team wants to estimate the probability that a user completes a target action as exposure increases. A saturating model, $P(x) = 1 - e^{-kx}$, captures diminishing returns. A linear baseline provides a useful comparison, but can become unrealistic near the boundaries.

In [ ]:
#Define probability models
def saturating_probability(exposure, rate):
    exposure = np.asarray(exposure)
    return 1 - np.exp(-rate * exposure)

def linear_probability(exposure, slope):
    exposure = np.asarray(exposure)
    return slope * exposure

#Generate exposure values and model parameters
exposure = np.linspace(0, 1, 200)
rate = 1.6
slope = 0.88

#Compute probabilities using the defined models
saturating = saturating_probability(exposure, rate)
linear = linear_probability(exposure, slope)

#Visualize the model comparison
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(exposure, saturating, label=f'Saturating model (k={rate})', linewidth=2)
ax.plot(exposure, linear, label=f'Linear baseline (slope={slope})', linewidth=2)
ax.set(xlabel='Normalised exposure', ylabel='Estimated probability', title='Model comparison')
ax.legend()
plt.show()

In [ ]:
#Evaluate models at specific points and compute required exposures
midpoint = 0.5
target = 0.75
saturating_at_midpoint = saturating_probability(midpoint, rate)
linear_at_midpoint = linear_probability(midpoint, slope)
exposure_for_target = -np.log1p(-target) / rate
linear_exposure_for_target = target / slope

#Print the computed values
print(f'Saturating model at x=0.5: {saturating_at_midpoint:.3f}')
print(f'Linear model at x=0.5: {linear_at_midpoint:.3f}')
print(f'Exposure for 0.75 under saturating model: {exposure_for_target:.3f}')
print(f'Exposure for 0.75 under linear model: {linear_exposure_for_target:.3f}')

#Perform assertions to validate model behavior
assert np.isclose(linear_at_midpoint, 0.44)
assert np.all((saturating >= 0) & (saturating <= 1))

The saturating model reaches the target with less exposure for these parameters, but that conclusion depends entirely on the selected rate. Neither curve establishes causality: they are transparent assumptions that would need observed data for calibration and validation.

## 2. Bayesian updating for a synthetic screening process

A quality team uses a binary screening test. We want the probability that an item truly belongs to the target class after receiving a positive result. The calculation combines prevalence, sensitivity, and the false-positive rate.

In [ ]:
#Define function to compute positive predictive value
def positive_predictive_value(prevalence, sensitivity, false_positive_rate):
    positive_probability = (sensitivity * prevalence) + (false_positive_rate * (1 - prevalence))
    return (sensitivity * prevalence) / positive_probability

#Define parameters for the test scenario
prevalence = 0.30
sensitivity = 0.92
false_positive_rate = 0.04

#Compute positive predictive value and probability of a positive result
positive_probability = (sensitivity * prevalence) + (false_positive_rate * (1 - prevalence))
ppv = positive_predictive_value(prevalence, sensitivity, false_positive_rate)

#Print the results
print(f'Probability of a positive result: {positive_probability:.3f}')
print(f'Probability an item is truly positive given a positive result: {ppv:.3f}')

In [ ]:
#Visualize how positive predictive value changes with prevalence
prevalence_grid = np.linspace(0.01, 0.99, 200)
ppv_grid = positive_predictive_value(prevalence_grid, sensitivity, false_positive_rate)

#Create the visualization
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(prevalence_grid, ppv_grid, color='tab:green', linewidth=2)
ax.scatter([prevalence], [ppv], color='black', zorder=3, label='Illustrative scenario')
ax.set(xlabel='Prevalence', ylabel='Positive predictive value', title='Prior prevalence changes the meaning of a positive test')
ax.legend()
plt.show()

A positive result is not the same as certainty. The posterior probability depends strongly on prevalence, which is why test performance should be communicated together with the population and decision context.

## 3. Sampling error and imperfect tests

For a binary proportion, the approximate standard error is $\sqrt{p(1-p)/n}$. The maximum occurs at $p=0.5$, so it gives a conservative planning value when the true proportion is not yet known.

In [ ]:
#Compute required sample size for a given target standard error
assumed_proportion = 0.5
target_standard_error = 0.025
required_sample = int(np.ceil(assumed_proportion * (1 - assumed_proportion) / target_standard_error**2))

#Print the computed sample size and approximate standard error
print(f'Conservative sample size: {required_sample}')
print(f'Approximate standard error: {np.sqrt(assumed_proportion * (1 - assumed_proportion) / required_sample):.3f}')
assert required_sample == 400

In [ ]:
#Define parameters for the test scenario
prior_rate = 0.50
false_negative_rate = 0.06
false_positive_rate = 0.03
sensitivity = 1 - false_negative_rate

#Compute observed positive rate and adjusted rate for true positives
observed_positive_rate = (sensitivity * prior_rate) + (false_positive_rate * (1 - prior_rate))
adjusted_rate = (sensitivity * prior_rate) / observed_positive_rate

#Print the results and perform assertions
print(f'Observed positive rate: {observed_positive_rate:.3f}')
print(f'Probability of a true positive after a positive test: {adjusted_rate:.3f}')
assert 0 < adjusted_rate < 1

### Limitations and next steps

These examples assume independent observations, fixed test characteristics, and correctly specified parameters. A production analysis would validate assumptions against representative data, quantify confidence or credible intervals, test sensitivity to parameter changes, and document the decision threshold.